# Projeto de Deep Learning: Classificação de Sons Ambientais com AST

Este notebook realiza o *fine-tuning* de um **Audio Spectrogram Transformer (AST)** no dataset **ESC-50**, composto por 2.000 áudios de 5 segundos distribuídos em 50 classes.

O pipeline está organizado em seis etapas:

1. instalação/importação e reprodutibilidade;
2. carregamento e divisão oficial do ESC-50;
3. reamostragem para 16 kHz e extração de espectrogramas Log-Mel;
4. configuração do AST por *transfer learning*;
5. treinamento e avaliação;
6. inferência e visualização.

> Execute as células em ordem. O treinamento é a única etapa demorada; ao final, o melhor modelo será salvo para reutilização.


## Fundamentos teóricos

### 1. Forma de onda e taxa de amostragem

Para o computador, um áudio é um vetor de amplitudes. A taxa de amostragem informa quantas medições existem por segundo: em 16 kHz, cada segundo contém 16.000 valores. Todos os áudios devem usar a taxa esperada pelo modelo para que tempo e frequência sejam interpretados corretamente.

### 2. Do domínio do tempo ao domínio da frequência

A Transformada de Fourier separa o sinal em componentes de frequência. Aplicando filtros na escala Mel e o logaritmo da energia, obtemos o espectrograma Log-Mel: tempo no eixo X, frequência no eixo Y e intensidade representada pelas cores.

### 3. Audio Spectrogram Transformer

O AST divide o espectrograma em pequenos *patches*. Cada patch é convertido em um vetor e processado por camadas de autoatenção, que aprendem relações entre diferentes regiões de tempo e frequência. Neste projeto, usamos *transfer learning*: os pesos aprendidos no AudioSet são reaproveitados e a camada final é adaptada para as 50 classes do ESC-50.


## 1. Instalação, importações e reprodutibilidade

No ambiente local, as dependências estão registradas em `requirements.txt` e instaladas na `.venv`. O FFmpeg é a dependência de sistema usada pelo TorchCodec para decodificar os áudios. Para recriar o ambiente no macOS, execute `python3.11 -m venv .venv`, `source .venv/bin/activate`, `pip install -r requirements.txt` e `brew install ffmpeg`.

O áudio é originalmente uma sequência unidimensional de amplitudes. O ESC-50 usa 44,1 kHz, mas o AST pré-treinado espera entradas em 16 kHz. A semente fixa torna a execução mais reproduzível.


In [ ]:
from pathlib import Path
import random

import evaluate
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchaudio

from datasets import Audio, DatasetDict, load_dataset
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from transformers import (
    ASTForAudioClassification,
    AutoConfig,
    AutoFeatureExtractor,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_ID = "MIT/ast-finetuned-audioset-10-10-0.4593"
OUTPUT_DIR = Path("./resultados_ast_esc50_corrigido")
FINAL_MODEL_DIR = Path("./modelo_ast_esc50_final_corrigido")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("PyTorch:", torch.__version__)
print("Dispositivo:", DEVICE)


## 2. Carregamento e divisão oficial do ESC-50

O ESC-50 possui cinco *folds* oficiais e balanceados. Neste experimento, os folds 1–4 são usados para treino e o fold 5 para teste. Assim, cada classe possui 32 exemplos de treino e 8 de teste, evitando uma divisão aleatória fora do protocolo do dataset.


In [ ]:
dataset_completo = load_dataset("ashraq/esc50")["train"]

print(dataset_completo)
print("Colunas:", dataset_completo.column_names)
print("Quantidade de classes:", len(set(dataset_completo["target"])))


### Reamostragem correta para 16 kHz

A coluna de áudio é convertida com `Audio(sampling_rate=16000)`. Isso altera cada clipe de 220.500 amostras em 44,1 kHz para aproximadamente 80.000 amostras em 16 kHz, preservando seus 5 segundos.


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

dataset_completo = dataset_completo.cast_column(
    "audio",
    Audio(sampling_rate=feature_extractor.sampling_rate),
)

dataset = DatasetDict({
    "train": dataset_completo.filter(
        lambda exemplo: exemplo["fold"] != 5,
        desc="Separando folds 1-4 para treino",
    ),
    "test": dataset_completo.filter(
        lambda exemplo: exemplo["fold"] == 5,
        desc="Separando fold 5 para teste",
    ),
})

audio_verificacao = dataset["train"][0]["audio"]
assert audio_verificacao["sampling_rate"] == feature_extractor.sampling_rate
assert len(audio_verificacao["array"]) == 5 * feature_extractor.sampling_rate

print(dataset)
print("Taxa após reamostragem:", audio_verificacao["sampling_rate"], "Hz")
print("Amostras em 5 segundos:", len(audio_verificacao["array"]))


### Classes e balanceamento

O mapeamento entre identificadores numéricos e nomes será salvo na configuração do modelo. Isso permite transformar diretamente a saída numérica do AST em uma categoria legível.


In [ ]:
pares_unicos = set(zip(dataset["train"]["target"], dataset["train"]["category"]))
pares_ordenados = sorted(pares_unicos, key=lambda par: par[0])

labels = [categoria for _, categoria in pares_ordenados]
num_labels = len(labels)
label2id = {label: indice for indice, label in enumerate(labels)}
id2label = {indice: label for indice, label in enumerate(labels)}

assert num_labels == 50

contagens_treino = np.bincount(dataset["train"]["target"], minlength=num_labels)
contagens_teste = np.bincount(dataset["test"]["target"], minlength=num_labels)

print("Classes:", num_labels)
print("Exemplos por classe no treino:", np.unique(contagens_treino))
print("Exemplos por classe no teste:", np.unique(contagens_teste))
print(pares_ordenados)


## 3. Pré-processamento com o Feature Extractor

O `AutoFeatureExtractor` converte os vetores de áudio, já reamostrados para 16 kHz, em bancos de filtros Log-Mel. O AST usa esses valores como uma representação bidimensional de tempo e frequência.

O comprimento padrão de 1.024 frames do modelo é mantido. Como os áudios possuem 5 segundos, o extrator preenche o restante da entrada com zeros.


In [ ]:
def preprocess_function(exemplos):
    audio_arrays = [
        np.asarray(audio["array"], dtype=np.float32)
        for audio in exemplos["audio"]
    ]

    return feature_extractor(
        audio_arrays,
        sampling_rate=feature_extractor.sampling_rate,
    )


colunas_para_remover = [
    coluna
    for coluna in dataset["train"].column_names
    if coluna != "target"
]

encoded_dataset = dataset.map(
    preprocess_function,
    batched=True,
    batch_size=16,
    remove_columns=colunas_para_remover,
    desc="Extraindo espectrogramas Log-Mel",
)

encoded_dataset = encoded_dataset.rename_column("target", "labels")
encoded_dataset.set_format("torch")

print(encoded_dataset)
print("Formato de uma entrada:", tuple(encoded_dataset["train"][0]["input_values"].shape))


## 4. Configuração do AST por Transfer Learning

O modelo parte de pesos treinados no AudioSet. A cabeça de classificação original é substituída por uma nova camada com 50 saídas, uma para cada categoria do ESC-50.


In [ ]:
config = AutoConfig.from_pretrained(
    MODEL_ID,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
)

model = ASTForAudioClassification.from_pretrained(
    MODEL_ID,
    config=config,
    ignore_mismatched_sizes=True,
)

parametros_totais = sum(parametro.numel() for parametro in model.parameters())
parametros_treinaveis = sum(
    parametro.numel()
    for parametro in model.parameters()
    if parametro.requires_grad
)

print("Modelo configurado com sucesso!")
print(f"Parâmetros totais: {parametros_totais:,}")
print(f"Parâmetros treináveis: {parametros_treinaveis:,}")


## 5. Fine-tuning e avaliação

A acurácia mede a proporção total de previsões corretas. O F1 macro calcula o F1 de cada classe e depois tira a média, dando o mesmo peso às 50 categorias.


In [ ]:
accuracy_metric = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=eval_pred.label_ids,
    )["accuracy"]

    relatorio = classification_report(
        eval_pred.label_ids,
        predictions,
        output_dict=True,
        zero_division=0,
    )

    return {
        "accuracy": accuracy,
        "f1_macro": relatorio["macro avg"]["f1-score"],
    }


### Treinamento

Esta célula pode demorar cerca de uma hora no Mac. Ao terminar, o melhor modelo é carregado automaticamente e salvo em `modelo_ast_esc50_final_corrigido`.

> Depois que o modelo estiver salvo, não execute novamente esta célula. Em sessões futuras, pule diretamente para a seção “Carregamento do modelo salvo”.


In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    warmup_steps=30,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    seed=SEED,
    data_seed=SEED,
    dataloader_pin_memory=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["test"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
)

print("Iniciando o treinamento...")
resultado_treino = trainer.train()

trainer.save_model(str(FINAL_MODEL_DIR))
feature_extractor.save_pretrained(str(FINAL_MODEL_DIR))
trainer.save_state()

print(f"Treinamento concluído em {resultado_treino.metrics['train_runtime'] / 60:.1f} minutos.")
print("Melhor checkpoint:", trainer.state.best_model_checkpoint)
print("Modelo final salvo em:", FINAL_MODEL_DIR.resolve())


### Avaliação final e matriz de confusão

A matriz de confusão normalizada mostra quais classes são confundidas entre si. A diagonal representa os acertos; valores fora dela indicam os principais erros do modelo.


In [ ]:
metricas_finais = trainer.evaluate()
print("Métricas finais:")
for nome, valor in metricas_finais.items():
    if isinstance(valor, float):
        print(f"  {nome}: {valor:.4f}")

resultado_predicoes = trainer.predict(encoded_dataset["test"])
y_true = resultado_predicoes.label_ids
y_pred = np.argmax(resultado_predicoes.predictions, axis=1)

print("\nRelatório por classe:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=list(range(num_labels)),
        target_names=labels,
        digits=3,
        zero_division=0,
    )
)

fig, ax = plt.subplots(figsize=(18, 16))
ConfusionMatrixDisplay.from_predictions(
    y_true,
    y_pred,
    labels=list(range(num_labels)),
    display_labels=labels,
    normalize="true",
    include_values=False,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax,
    colorbar=False,
)
ax.set_title("Matriz de confusão normalizada — ESC-50")
plt.tight_layout()
plt.show()


In [ ]:
historico_avaliacao = [
    registro
    for registro in trainer.state.log_history
    if "eval_accuracy" in registro
]

epocas = [registro["epoch"] for registro in historico_avaliacao]
acuracias = [registro["eval_accuracy"] for registro in historico_avaliacao]
perdas = [registro["eval_loss"] for registro in historico_avaliacao]

fig, eixos = plt.subplots(1, 2, figsize=(12, 4))

eixos[0].plot(epocas, acuracias, marker="o")
eixos[0].set_title("Acurácia de validação")
eixos[0].set_xlabel("Época")
eixos[0].set_ylabel("Acurácia")
eixos[0].grid(alpha=0.3)

eixos[1].plot(epocas, perdas, marker="o", color="tab:red")
eixos[1].set_title("Loss de validação")
eixos[1].set_xlabel("Época")
eixos[1].set_ylabel("Loss")
eixos[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## Carregamento do modelo salvo

Em uma nova sessão, execute a célula de importações e esta célula. Não é necessário repetir o treinamento. Para avaliar novamente o dataset completo, também execute as etapas de carregamento e pré-processamento.


In [ ]:
feature_extractor = AutoFeatureExtractor.from_pretrained(FINAL_MODEL_DIR)
model = ASTForAudioClassification.from_pretrained(FINAL_MODEL_DIR)
model.to(DEVICE)
model.eval()

print("Modelo carregado de:", FINAL_MODEL_DIR.resolve())


## 6. Inferência e visualização

Primeiro, o modelo é testado em um exemplo do fold de teste. São mostradas a classe correta, a previsão e as cinco categorias mais prováveis.


In [ ]:
def predict_audio(audio_array, sampling_rate, top_k=5):
    audio_array = np.asarray(audio_array, dtype=np.float32)

    if sampling_rate != feature_extractor.sampling_rate:
        audio_array = librosa.resample(
            audio_array,
            orig_sr=sampling_rate,
            target_sr=feature_extractor.sampling_rate,
        )
        sampling_rate = feature_extractor.sampling_rate

    inputs = feature_extractor(
        audio_array,
        sampling_rate=sampling_rate,
        return_tensors="pt",
    )
    inputs = {
        nome: tensor.to(DEVICE)
        for nome, tensor in inputs.items()
    }

    model.eval()
    with torch.no_grad():
        logits = model(**inputs).logits
        probabilidades = torch.softmax(logits, dim=-1)[0]

    valores, indices = torch.topk(probabilidades, k=top_k)

    return [
        {
            "classe_id": indice.item(),
            "classe": model.config.id2label[indice.item()],
            "probabilidade": valor.item(),
        }
        for valor, indice in zip(valores.cpu(), indices.cpu())
    ]


exemplo = dataset["test"][0]
audio_teste = exemplo["audio"]

previsoes = predict_audio(
    audio_teste["array"],
    audio_teste["sampling_rate"],
)

print("Classe correta:", exemplo["category"])
print("Classe prevista:", previsoes[0]["classe"])
print("\nTop 5 previsões:")
for previsao in previsoes:
    print(f"  {previsao['classe']:<25} {previsao['probabilidade']:.2%}")


### Espectrograma Log-Mel

O eixo horizontal representa o tempo, o eixo vertical representa as frequências na escala Mel e as cores representam a energia em decibéis. Esta visualização é conceitualmente equivalente à representação usada pelo AST; internamente, o extrator do modelo utiliza filtros Mel compatíveis com Kaldi.


In [ ]:
audio_array = np.asarray(audio_teste["array"], dtype=np.float32)
sampling_rate = audio_teste["sampling_rate"]

espectrograma_mel = librosa.feature.melspectrogram(
    y=audio_array,
    sr=sampling_rate,
    n_fft=1024,
    hop_length=160,
    n_mels=128,
)
espectrograma_db = librosa.power_to_db(
    espectrograma_mel,
    ref=np.max,
)

plt.figure(figsize=(12, 5))
librosa.display.specshow(
    espectrograma_db,
    sr=sampling_rate,
    hop_length=160,
    x_axis="time",
    y_axis="mel",
    cmap="magma",
)
plt.colorbar(format="%+2.0f dB")
plt.title(
    f"Espectrograma Log-Mel — real: {exemplo['category']} | "
    f"previsto: {previsoes[0]['classe']}"
)
plt.tight_layout()
plt.show()


### Teste com um áudio próprio

Coloque um arquivo WAV, MP3 ou FLAC na pasta `notebooks` e altere o nome abaixo. O `librosa.load(..., sr=16000)` realiza a reamostragem correta antes da inferência.


In [ ]:
ARQUIVO_AUDIO = Path("./meu_audio.wav")

if ARQUIVO_AUDIO.exists():
    audio_proprio, sr_proprio = librosa.load(
        ARQUIVO_AUDIO,
        sr=feature_extractor.sampling_rate,
        mono=True,
    )
    previsoes_audio_proprio = predict_audio(audio_proprio, sr_proprio)

    print("Previsão:", previsoes_audio_proprio[0]["classe"])
    print("\nTop 5:")
    for previsao in previsoes_audio_proprio:
        print(f"  {previsao['classe']:<25} {previsao['probabilidade']:.2%}")
else:
    print(
        f"Arquivo não encontrado: {ARQUIVO_AUDIO}. "
        "Altere ARQUIVO_AUDIO para testar um som próprio."
    )


## Conclusões e pontos para a apresentação

- O ESC-50 foi reamostrado de 44,1 kHz para 16 kHz antes da extração de características.
- A divisão respeitou os folds oficiais: folds 1–4 para treino e fold 5 para teste.
- O AST converte o Log-Mel em patches e usa autoatenção para relacionar regiões do espectrograma.
- O *transfer learning* reaproveitou representações aprendidas no AudioSet e adaptou a camada final para 50 classes.
- Acurácia, F1 macro, relatório por classe e matriz de confusão foram usados na avaliação.
- O modelo final foi salvo e pode classificar novos arquivos sem repetir o treinamento.

**Limitação:** este notebook avalia apenas o fold 5. O protocolo completo do ESC-50 treina cinco modelos, alternando o fold de teste, e relata a média dos cinco resultados. Para o prazo do projeto, um único fold oficial oferece uma avaliação clara, desde que essa limitação seja declarada.
